# 04. Pandas para sociolingüística

En este ejercicio vamos a repasar [pandas](https://pandas.pydata.org/), la biblioteca de Python más usada para trabajar con **datos tabulares** (tablas con filas y columnas, como una hoja de cálculo).

Vamos a ir de lo más básico a lo más elaborado:

1. Cargar archivos
2. Limpiar datos
3. Juntar DataFrames
4. Separar columnas y DataFrames
5. Seleccionar y filtrar
6. Agrupar y resumir
7. Visualizar
8. Ejercicio final

Cada sección tiene **ejemplos** (celdas que ya funcionan: ejecútalas y lee el resultado) y **ejercicios** (celdas con `# Tu código aquí`).

## El estudio (ficticio)

Imagina que formamos parte de un proyecto que estudia la **pluralización de *haber* existencial** en el español de México:

| Variante singular (normativa) | Variante plural (no normativa) |
|---|---|
| *Había muchas personas en la fiesta.* | *Habían muchas personas en la fiesta.* |
| *Hubo dos accidentes.* | *Hubieron dos accidentes.* |
| *Ha habido varios problemas.* | *Han habido varios problemas.* |

En términos sociolingüísticos, *haber* existencial es nuestra **variable dependiente** y tiene dos **variantes**: singular y plural. Queremos saber qué factores favorecen la variante plural:

- **Factores sociales:** ciudad, edad, sexo y escolaridad de la persona que habla.
- **Factores lingüísticos:** tiempo verbal de *haber*.
- **Factores estilísticos:** registro (conversación o entrevista).

Tres equipos recogieron datos en tres ciudades: Ciudad de México (CDMX), Guadalajara (GDL) y Monterrey (MTY). Cada equipo entregó sus archivos a su manera... así que vamos a tener que limpiarlos.

> ⚠️ **Los datos son simulados.** Sirven para practicar pandas; los resultados **no** describen el español real de esas ciudades.

### Archivos (en `datos/haber_existencial/`)

| Archivo | Contenido | Formato |
|---|---|---|
| `hablantes.csv` | Una fila por persona entrevistada (metadatos sociales) | CSV separado por comas, UTF-8 |
| `corpus_cdmx.csv` | Una fila por ocurrencia de *haber* existencial en CDMX | CSV separado por comas, UTF-8 |
| `corpus_gdl.tsv` | Ocurrencias de Guadalajara | Separado por tabuladores, UTF-8 |
| `corpus_mty.csv` | Ocurrencias de Monterrey | Separado por `;`, codificación Latin-1 (exportado desde Excel) |

Para la columna `Sexo` de `hablantes.csv`, el equipo usó estas etiquetas: `M` = mujer, `F` = femenino (mujer), `H` = hombre.

## 0. Preparación

Si no tienes pandas y matplotlib instalados, ejecuta en la terminal (o en una celda, anteponiendo `!`):

```bash
pip install pandas matplotlib
```

Por convención, pandas se importa como `pd`. Este notebook debe ejecutarse desde la raíz del repositorio para que funcionen las rutas.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATOS = Path("datos/haber_existencial")
RESULTADOS = Path("resultados")
RESULTADOS.mkdir(exist_ok=True)

pd.__version__

## 1. Cargar archivos

La función principal para leer tablas es `pd.read_csv()`. Regresa un **DataFrame**: una tabla donde cada columna es una **Series** (una columna con nombre y un tipo de dato).

In [ ]:
hablantes = pd.read_csv(DATOS / "hablantes.csv")
hablantes.head()  # las primeras 5 filas

### Primer vistazo

Antes de hacer cualquier análisis, **siempre** revisa qué cargaste:

| Código | ¿Qué te dice? |
|---|---|
| `df.shape` | (número de filas, número de columnas) |
| `df.columns` | los nombres de las columnas |
| `df.info()` | tipos de dato y cuántos valores no nulos hay por columna |
| `df.head(n)`, `df.tail(n)`, `df.sample(n)` | algunas filas (al principio, al final, al azar) |
| `df.describe()` | estadística descriptiva de las columnas numéricas |
| `df["columna"].value_counts()` | cuántas veces aparece cada valor en una columna |

In [ ]:
hablantes.shape

In [ ]:
hablantes.columns

¿Notas algo raro en los nombres de las columnas? Fíjate en `' Edad'`: tiene un espacio al principio. Si escribes `hablantes["Edad"]` obtendrás un `KeyError`. Lo arreglaremos en la sección 2.

In [ ]:
hablantes.info()

`info()` dice que la edad **no** es numérica. ¿Por qué? Veamos algunos valores. Para seleccionar una columna se usan corchetes con su nombre:

In [ ]:
hablantes[" Edad"].head(10)

In [ ]:
hablantes["Sexo"].value_counts()

La misma categoría aparece escrita de muchas formas (`Mujer`, `mujer`, `Mujer ` con espacio, `M`, `F`...). Para la computadora son valores distintos. Esto es muy común cuando varias personas capturan datos a mano.

### Otros separadores y codificaciones

No todos los archivos "CSV" están separados por comas ni usan UTF-8:

- `sep="\t"` para archivos separados por tabuladores (`.tsv`).
- `sep=";"` para archivos exportados por Excel en configuraciones en español.
- `encoding="latin-1"` para archivos con otra codificación. Si ves un `UnicodeDecodeError` o caracteres como `Ã­` en lugar de `í`, el problema es la codificación.

Mira lo que pasa si intentamos leer el archivo de Monterrey sin indicar nada:

In [ ]:
try:
    pd.read_csv(DATOS / "corpus_mty.csv")
except UnicodeDecodeError as error:
    print("No se pudo leer el archivo:", error)

### ✏️ Ejercicio 1

1. Carga los tres corpus en las variables `cdmx`, `gdl` y `mty`. Elige el separador y la codificación correctos para cada uno.
2. Para cada uno: ¿cuántas filas y columnas tiene? ¿Cómo se llaman sus columnas?
3. ¿Qué diferencia hay entre las columnas de `mty` y las de los otros dos corpus? ¿Por qué podría ser un problema más adelante?
4. Usa `value_counts()` sobre la columna de la forma verbal de `cdmx`. ¿Ves formas que deberían ser iguales pero aparecen separadas?

In [ ]:
# Tu código aquí

## 2. Limpiar datos

Limpiar datos es casi siempre la parte que más tiempo toma. Vamos a limpiar `hablantes` paso a paso.

### 2.1 Nombres de columnas

Una buena práctica es usar nombres en minúsculas, sin espacios ni acentos. Las columnas tienen métodos de texto en `.str`:

In [ ]:
hablantes.columns = (
    hablantes.columns
    .str.strip()             # quita espacios al principio y al final
    .str.lower()             # pasa a minúsculas
    .str.replace(" ", "_")   # cambia espacios internos por guion bajo
)
hablantes.columns

También podemos renombrar columnas específicas con `rename()`:

In [ ]:
hablantes = hablantes.rename(columns={"id_hablante": "id"})
hablantes.columns

### 2.2 Filas duplicadas

`duplicated()` marca con `True` las filas que son copia exacta de una anterior. Como `True` vale 1, `.sum()` las cuenta.

In [ ]:
hablantes.duplicated().sum()

In [ ]:
hablantes[hablantes.duplicated(keep=False)]  # muestra todas las copias

In [ ]:
hablantes = hablantes.drop_duplicates()
hablantes.shape

También es importante revisar que el **identificador** no se repita (aunque el resto de la fila sea distinta):

In [ ]:
hablantes["id"].is_unique

### 2.3 Texto: espacios, mayúsculas y categorías

Los métodos de `.str` funcionan sobre todos los valores de una columna a la vez:

In [ ]:
for columna in ["sexo", "escolaridad", "ciudad"]:
    hablantes[columna] = hablantes[columna].str.strip().str.lower()

hablantes["sexo"].value_counts()

Para unificar categorías usamos `replace()` con un diccionario `{valor_viejo: valor_nuevo}`. Los valores que no están en el diccionario se quedan igual.

In [ ]:
hablantes["sexo"] = hablantes["sexo"].replace({"m": "mujer", "f": "mujer", "h": "hombre"})
hablantes["sexo"].value_counts()

In [ ]:
hablantes["escolaridad"].value_counts()

In [ ]:
hablantes["escolaridad"] = hablantes["escolaridad"].replace({
    "prepa": "preparatoria",
    "bachillerato": "preparatoria",
    "licenciatura": "universidad",
})
hablantes["escolaridad"].value_counts()

### 2.4 Números guardados como texto

Algunas edades dicen `"35 años"`. Con `str.extract()` y una **expresión regular** sacamos sólo los dígitos (`\d+`), y con `pd.to_numeric()` los convertimos en números.

In [ ]:
hablantes["edad"] = pd.to_numeric(hablantes["edad"].str.extract(r"(\d+)", expand=False))
hablantes["edad"].describe()

### 2.5 Valores faltantes

En pandas, un dato faltante aparece como `NaN` (o `<NA>`). `isna()` regresa `True` donde falta un valor.

In [ ]:
hablantes.isna().sum()

In [ ]:
hablantes[hablantes["edad"].isna()]

¿Qué hacemos con los faltantes? Hay varias opciones:

- `dropna()` elimina las filas con valores faltantes.
- `fillna(valor)` rellena con un valor (por ejemplo, la mediana).
- Dejarlos como están y recordarlo al analizar.

En sociolingüística **inventar** la edad de una persona (rellenar con la mediana) puede sesgar los resultados. Aquí los dejaremos como `NaN`: pandas los ignora al calcular promedios. Lo ideal sería volver a las fichas de las entrevistas para recuperar el dato.

### ✏️ Ejercicio 2

Ahora limpia la columna `ciudad` de `hablantes` y los tres corpus.

1. Unifica los valores de `hablantes["ciudad"]` para que sólo queden tres: `"CDMX"`, `"GDL"` y `"MTY"`.
2. En `mty`, cambia los nombres de las columnas a minúsculas para que coincidan con los de `cdmx` y `gdl`.
3. En los tres corpus, limpia la columna `forma`: quita espacios y pasa todo a minúsculas.
4. ¿Cuántas filas duplicadas tiene cada corpus? Elimínalas.
5. ¿Cuántas ocurrencias no tienen `forma`?
6. **Reto:** la forma verbal es la primera palabra del enunciado (o las dos primeras, en *ha habido* / *han habido*). Recupera las formas faltantes a partir de la columna `enunciado`. Pistas: `str.extract(r"^(\w+(?: habido)?)")` y `fillna()`.

In [ ]:
# Tu código aquí

## 3. Juntar DataFrames

Hay dos maneras principales de juntar tablas:

| Función | ¿Qué hace? | Analogía |
|---|---|---|
| `pd.concat()` | Pone tablas una **debajo** de otra (o una al lado de otra) | Pegar hojas de cálculo con las mismas columnas |
| `df.merge()` | Combina tablas usando una **columna en común** (una llave) | `JOIN` en SQL / `BUSCARV` en Excel |

### 🚩 Punto de control

La siguiente celda carga y limpia los tres corpus, para que puedas seguir aunque no hayas terminado los ejercicios 1 y 2. **Intenta resolverlos tú antes de leerla.**

In [ ]:
def cargar_corpus(ruta, **opciones):
    corpus = pd.read_csv(ruta, **opciones)
    corpus.columns = corpus.columns.str.strip().str.lower()
    corpus["forma"] = corpus["forma"].str.strip().str.lower()
    corpus["forma"] = corpus["forma"].fillna(
        corpus["enunciado"].str.extract(r"^(\w+(?: habido)?)", expand=False).str.lower()
    )
    return corpus.drop_duplicates()


cdmx = cargar_corpus(DATOS / "corpus_cdmx.csv")
gdl = cargar_corpus(DATOS / "corpus_gdl.tsv", sep="\t")
mty = cargar_corpus(DATOS / "corpus_mty.csv", sep=";", encoding="latin-1")

hablantes["ciudad"] = hablantes["ciudad"].replace({
    "ciudad de méxico": "CDMX", "ciudad de mexico": "CDMX", "cdmx": "CDMX",
    "guadalajara": "GDL", "gdl": "GDL",
    "monterrey": "MTY", "mty": "MTY",
})
hablantes["ciudad"].value_counts()

### 3.1 `pd.concat()`: una tabla debajo de otra

Los tres corpus tienen las mismas columnas, así que podemos apilarlos. `ignore_index=True` crea un índice nuevo (0, 1, 2...) en vez de repetir los índices de cada tabla.

In [ ]:
corpus = pd.concat([cdmx, gdl, mty], ignore_index=True)
print(len(cdmx), len(gdl), len(mty), "->", len(corpus))
corpus.sample(5, random_state=1)

⚠️ `concat()` alinea las columnas **por nombre**. Si los nombres no coinciden, aparecen columnas nuevas llenas de `NaN`. Esto es lo que habría pasado si no hubiéramos pasado a minúsculas los encabezados de Monterrey:

In [ ]:
mty_sin_limpiar = pd.read_csv(DATOS / "corpus_mty.csv", sep=";", encoding="latin-1")
pd.concat([cdmx, mty_sin_limpiar], ignore_index=True).columns

### 3.2 `merge()`: combinar por una llave

El corpus tiene una fila por **ocurrencia** y sólo dice quién la produjo (`informante`). La edad, el sexo y la escolaridad están en `hablantes`. Con `merge()` añadimos a cada ocurrencia los datos de quien la dijo.

Como la llave se llama distinto en cada tabla (`informante` y `id`), usamos `left_on` y `right_on`:

In [ ]:
datos = corpus.merge(hablantes, left_on="informante", right_on="id", how="left")
datos.head()

El parámetro `how` decide qué pasa con las filas que no encuentran pareja:

| `how=` | Conserva... |
|---|---|
| `"inner"` | sólo las filas con pareja en **ambas** tablas |
| `"left"` | todas las filas de la tabla izquierda (`corpus`) |
| `"right"` | todas las filas de la tabla derecha (`hablantes`) |
| `"outer"` | todas las filas de ambas tablas |

Con `indicator=True` pandas agrega una columna `_merge` que dice de dónde vino cada fila. Es muy útil para detectar problemas:

In [ ]:
revision = corpus.merge(hablantes, left_on="informante", right_on="id", how="outer", indicator=True)
revision["_merge"].value_counts()

In [ ]:
revision.loc[revision["_merge"] != "both", ["id_ocurrencia", "informante", "enunciado", "_merge"]]

Hay ocurrencias de un informante (`GDL-099`) que no aparece en la tabla de hablantes. No sabemos su edad, sexo ni escolaridad.

Por último, `validate="many_to_one"` le pide a pandas que verifique que cada informante aparece **una sola vez** en `hablantes`. Si no hubiéramos borrado los duplicados en la sección 2, esta línea daría un error (¡y las ocurrencias se habrían contado dos veces!).

In [ ]:
datos = corpus.merge(hablantes, left_on="informante", right_on="id", how="inner", validate="many_to_one")
datos = datos.drop(columns="id")
datos.shape

### ✏️ Ejercicio 3

1. Vuelve a concatenar los tres corpus, ahora con `keys=["CDMX", "GDL", "MTY"]` y **sin** `ignore_index`. ¿Cómo cambia el índice? (Pista: después prueba `.reset_index(level=0)`).
2. ¿Cuántas filas obtienes al hacer el `merge` con `how="left"` y cuántas con `how="inner"`? Explica la diferencia.
3. Haz un `merge` con `how="right"`. ¿Hay hablantes sin ninguna ocurrencia?
4. ¿Qué harías con las ocurrencias de `GDL-099` en un proyecto real? Escribe tu respuesta en una celda de Markdown.

In [ ]:
# Tu código aquí

## 4. Separar

### 4.1 Separar una columna en varias

El código de informante (`CDMX-001`) contiene dos datos: la ciudad y un número. `str.split()` con `expand=True` regresa un DataFrame con una columna por pedazo:

In [ ]:
datos["informante"].str.split("-", expand=True).head()

In [ ]:
datos[["ciudad_codigo", "num_informante"]] = datos["informante"].str.split("-", expand=True)
datos[["informante", "ciudad_codigo", "num_informante"]].head()

Esto nos permite **verificar** los datos: la ciudad del código debería coincidir con la ciudad registrada en la tabla de hablantes.

In [ ]:
(datos["ciudad_codigo"] == datos["ciudad"]).all()

### 4.2 Separar un DataFrame en varios

Con una **condición** (una Series de `True`/`False`) nos quedamos sólo con las filas que la cumplen:

In [ ]:
conversacion = datos[datos["registro"] == "conversación"]
entrevista = datos[datos["registro"] == "entrevista"]
len(conversacion), len(entrevista)

Si queremos un DataFrame por cada valor de una columna, `groupby()` hace la separación por nosotros. Aquí guardamos un archivo por ciudad con `to_csv()`:

In [ ]:
for ciudad, subtabla in datos.groupby("ciudad"):
    ruta = RESULTADOS / f"datos_{ciudad.lower()}.csv"
    subtabla.to_csv(ruta, index=False)
    print(ruta, len(subtabla), "filas")

### ✏️ Ejercicio 4

1. Separa la columna `enunciado` en dos columnas: la primera palabra y el resto (pista: `str.split(" ", n=1, expand=True)`). ¿Funciona bien para *ha habido*? ¿Por qué?
2. Separa `datos` en un DataFrame por tiempo verbal y guarda cada uno en `resultados/` con un nombre como `tiempo_imperfecto.csv`.
3. Revisa que la suma de filas de todos los archivos que guardaste sea igual al número de filas de `datos`.

In [ ]:
# Tu código aquí

## 5. Seleccionar y filtrar

### 5.1 Columnas

In [ ]:
datos["forma"].head()                    # una columna -> Series

In [ ]:
datos[["informante", "forma", "tiempo"]].head()   # varias columnas -> DataFrame (¡doble corchete!)

### 5.2 Filas con `loc` e `iloc`

- `iloc[filas, columnas]` selecciona por **posición** (como en las listas de Python).
- `loc[filas, columnas]` selecciona por **etiqueta** o por **condición**.

In [ ]:
datos.iloc[0:3, 0:4]    # filas 0 a 2, columnas 0 a 3

In [ ]:
datos.loc[datos["ciudad"] == "MTY", ["informante", "enunciado", "escolaridad"]].head()

### 5.3 Condiciones

| Operación | Código |
|---|---|
| igual / distinto | `df["col"] == x`, `df["col"] != x` |
| mayor / menor | `df["col"] > x`, `df["col"].between(a, b)` |
| y / o / no | `&`, `\|`, `~` (¡cada condición entre paréntesis!) |
| pertenece a una lista | `df["col"].isin([x, y, z])` |
| contiene un texto | `df["col"].str.contains("texto")` |

Ejemplo: mujeres menores de 35 años en Guadalajara.

In [ ]:
filtro = (datos["sexo"] == "mujer") & (datos["edad"] < 35) & (datos["ciudad"] == "GDL")
datos.loc[filtro, ["informante", "edad", "enunciado"]].head()

`query()` permite escribir lo mismo de forma más legible:

In [ ]:
datos.query("sexo == 'mujer' and edad < 35 and ciudad == 'GDL'").shape

Para ordenar se usa `sort_values()`:

In [ ]:
hablantes.sort_values("edad", ascending=False).head(3)

### 5.4 Crear columnas nuevas

Ahora creamos la columna más importante del estudio: la **variante**. Con `isin()` revisamos si la forma es plural. Guardamos el resultado como 0/1 porque así **el promedio es la proporción** de plurales.

In [ ]:
PLURALES = ["habían", "hubieron", "han habido", "habrán", "habrían"]

datos["plural"] = datos["forma"].isin(PLURALES).astype(int)
datos["variante"] = datos["plural"].map({1: "plural", 0: "singular"})
datos[["forma", "plural", "variante"]].sample(5, random_state=2)

Para estudiar la edad en **tiempo aparente**, los sociolingüistas suelen agrupar a los hablantes en generaciones. `pd.cut()` convierte un número en rangos:

In [ ]:
datos["generacion"] = pd.cut(datos["edad"], bins=[17, 34, 54, 100], labels=["18-34", "35-54", "55+"])
datos[["edad", "generacion"]].head()

La escolaridad tiene un **orden** natural. Si se lo decimos a pandas, las tablas y gráficas lo respetarán (en lugar de usar orden alfabético):

In [ ]:
ORDEN_ESCOLARIDAD = ["primaria", "secundaria", "preparatoria", "universidad"]
datos["escolaridad"] = pd.Categorical(datos["escolaridad"], categories=ORDEN_ESCOLARIDAD, ordered=True)
datos["escolaridad"].dtype

### ✏️ Ejercicio 5

1. Selecciona las ocurrencias de hablantes con universidad de Monterrey en pretérito imperfecto. ¿Cuántas son plurales?
2. Muestra los enunciados de hombres mayores de 55 años que contengan la palabra *personas*.
3. ¿Qué hablante es la persona más joven del corpus? ¿Cuántas ocurrencias produjo?
4. Crea una columna `numero_sn` que diga si el sintagma nominal empieza con un numeral (*dos*, *tres*) o no. Pista: `str.contains(r"^\w+ (?:habido )?(?:dos|tres) ")`.

In [ ]:
# Tu código aquí

## 6. Agrupar y resumir

`value_counts(normalize=True)` da proporciones en lugar de conteos:

In [ ]:
datos["variante"].value_counts(normalize=True)

`groupby()` separa los datos en grupos, aplica una función a cada grupo y junta los resultados (*split-apply-combine*). Como `plural` vale 0 o 1, su promedio es la proporción de plurales:

In [ ]:
datos.groupby("ciudad")["plural"].mean()

Con `agg()` calculamos varias cosas a la vez. Siempre reporta **cuántos datos** hay detrás de cada proporción:

In [ ]:
datos.groupby(["ciudad", "escolaridad"], observed=True).agg(
    ocurrencias=("plural", "size"),
    plurales=("plural", "sum"),
    proporcion=("plural", "mean"),
).round(2)

`pd.crosstab()` hace tablas de contingencia (como las que se usan para una prueba de χ²). Con `normalize="index"` cada fila suma 1:

In [ ]:
pd.crosstab(datos["tiempo"], datos["variante"], normalize="index").round(2)

`pivot_table()` acomoda un resumen en formato ancho: un factor en las filas y otro en las columnas.

In [ ]:
tabla_esc_ciudad = datos.pivot_table(
    index="escolaridad", columns="ciudad", values="plural", aggfunc="mean", observed=True
)
tabla_esc_ciudad.round(2)

#### Un detalle metodológico

Cada hablante produjo un número distinto de ocurrencias, y las ocurrencias de una misma persona no son independientes. Por eso conviene calcular también la proporción **por hablante**:

In [ ]:
por_hablante = (
    datos.groupby(["informante", "ciudad", "sexo", "edad", "escolaridad"], observed=True)
    .agg(ocurrencias=("plural", "size"), proporcion=("plural", "mean"))
    .reset_index()
)
por_hablante.head()

### ✏️ Ejercicio 6

1. Calcula la proporción de plurales por tiempo verbal y ordénala de mayor a menor.
2. Calcula la proporción de plurales por sexo y generación (dos factores a la vez).
3. ¿Qué cinco hablantes usan más la variante plural? ¿Cuántas ocurrencias tiene cada uno?
4. ¿Cambia la proporción de plurales entre conversación y entrevista? ¿Qué esperarías desde la teoría de la variación estilística?

In [ ]:
# Tu código aquí

## 7. Visualizar

Los DataFrames y las Series tienen un método `.plot()` que usa matplotlib por debajo. El argumento `kind` elige el tipo de gráfica (`"bar"`, `"barh"`, `"line"`, `"hist"`, `"box"`, `"scatter"`...).

Antes de graficar, pregúntate **qué quieres mostrar**:

| Pregunta | Gráfica |
|---|---|
| Comparar una cantidad entre categorías | barras |
| Comparar dos factores a la vez | barras agrupadas |
| Ver cómo se reparte un total | barras apiladas al 100 % |
| Ver una distribución | histograma o diagrama de caja |
| Ver una tendencia ordenada (generaciones) | líneas |

Usaremos siempre los mismos colores para cada ciudad, para que se reconozcan en todas las gráficas.

In [ ]:
COLORES_CIUDAD = {"CDMX": "#2a78d6", "GDL": "#eb6834", "MTY": "#1baf7a"}
COLORES_VARIANTE = {"singular": "#2a78d6", "plural": "#eb6834"}

### 7.1 Barras: proporción de plurales por ciudad

In [ ]:
por_ciudad = datos.groupby("ciudad")["plural"].mean()

ax = por_ciudad.plot(kind="bar", color=[COLORES_CIUDAD[c] for c in por_ciudad.index], rot=0)
ax.set_title("Proporción de haber pluralizado por ciudad")
ax.set_xlabel("")
ax.set_ylabel("Proporción de la variante plural")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### 7.2 Barras agrupadas: escolaridad y ciudad

Cuando graficamos un DataFrame, cada **columna** se vuelve una serie (un color) y cada **fila** un grupo en el eje x. Por eso usamos la tabla de `pivot_table()` de la sección 6.

In [ ]:
ax = tabla_esc_ciudad.plot(kind="bar", color=COLORES_CIUDAD, rot=0, width=0.75)
ax.set_title("Proporción de haber pluralizado por escolaridad y ciudad")
ax.set_xlabel("Escolaridad")
ax.set_ylabel("Proporción de la variante plural")
ax.set_ylim(0, 1)
ax.legend(title="Ciudad", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### 7.3 Barras apiladas al 100 %: tiempo verbal

In [ ]:
tabla_tiempo = pd.crosstab(datos["tiempo"], datos["variante"], normalize="index")
tabla_tiempo = tabla_tiempo.sort_values("plural")[["singular", "plural"]]

ax = tabla_tiempo.plot(kind="barh", stacked=True, color=COLORES_VARIANTE, edgecolor="white")
ax.set_title("Variante según el tiempo verbal de haber")
ax.set_xlabel("Proporción de ocurrencias")
ax.set_ylabel("")
ax.legend(title="Variante", frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncols=2)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### 7.4 Distribuciones: histograma y diagrama de caja

¿Cómo se distribuyen las edades de quienes entrevistamos? ¿Y la proporción de plurales por hablante según su escolaridad?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

hablantes["edad"].plot(kind="hist", bins=10, ax=ax1, color="#2a78d6", edgecolor="white")
ax1.set_title("Edad de las personas entrevistadas")
ax1.set_xlabel("Edad")
ax1.set_ylabel("Número de personas")

por_hablante.boxplot(column="proporcion", by="escolaridad", ax=ax2, grid=False)
ax2.set_title("Proporción de plurales por hablante")
ax2.set_xlabel("Escolaridad")
ax2.set_ylabel("Proporción de la variante plural")
fig.suptitle("")  # boxplot(by=...) agrega un título automático; lo quitamos

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 7.5 Líneas: tiempo aparente

Si las generaciones jóvenes usan más una variante que las mayores, podría tratarse de un **cambio en marcha** (o de un patrón propio de cierta edad). Cada ciudad es una línea:

In [ ]:
tabla_generacion = datos.pivot_table(
    index="generacion", columns="ciudad", values="plural", aggfunc="mean", observed=True
)

ax = tabla_generacion.plot(kind="line", marker="o", color=COLORES_CIUDAD, linewidth=2)
ax.set_title("Haber pluralizado por generación (tiempo aparente)")
ax.set_xlabel("Generación")
ax.set_ylabel("Proporción de la variante plural")
ax.set_ylim(0, 1)
ax.legend(title="Ciudad", frameon=False)
ax.spines[["top", "right"]].set_visible(False)

ax.figure.savefig(RESULTADOS / "tiempo_aparente.png", dpi=150, bbox_inches="tight")
plt.show()

`savefig()` guarda la figura en un archivo (`.png`, `.pdf`, `.svg`...), listo para usar en una presentación o un artículo.

### ✏️ Ejercicio 7

1. Haz una gráfica de barras horizontales de la proporción de plurales por tiempo verbal, ordenada de menor a mayor.
2. Haz una gráfica de barras agrupadas con la proporción de plurales por sexo (grupos) y generación (colores).
3. Con `por_hablante`, haz una gráfica de dispersión (`kind="scatter"`) de edad contra proporción de plurales. Pinta cada punto con el color de su ciudad. Pista: `c=por_hablante["ciudad"].map(COLORES_CIUDAD)`.
4. Guarda tus tres gráficas en la carpeta `resultados/`.

In [ ]:
# Tu código aquí

## 8. Ejercicio final

Escribe un **breve reporte** (en celdas de Markdown, con el código y las gráficas que lo respaldan) que responda:

> ¿Qué factores sociales, lingüísticos y estilísticos parecen favorecer la pluralización de *haber* existencial en estos datos?

Tu reporte debe incluir:

1. Una tabla con el número de hablantes y de ocurrencias por ciudad.
2. Al menos **tres hallazgos**, cada uno con una tabla o una gráfica.
3. Un párrafo de **limitaciones**: ¿qué problemas encontraste al limpiar los datos? ¿Qué decisiones tomaste (duplicados, datos faltantes, el informante `GDL-099`)? ¿Por qué no basta con comparar proporciones para afirmar que un factor es significativo?

**Reto:** busca qué es la regresión logística de efectos mixtos (como la que usan Rbrul o el paquete `lme4` de R) y explica por qué se usa en sociolingüística variacionista en lugar de sólo comparar porcentajes.

In [ ]:
# Tu código aquí

*Escribe aquí tu reporte.*

## Chuleta de pandas

| Tarea | Código |
|---|---|
| Leer un CSV | `pd.read_csv(ruta, sep=",", encoding="utf-8")` |
| Primer vistazo | `df.head()`, `df.shape`, `df.info()`, `df.describe()` |
| Contar valores | `df["col"].value_counts(normalize=False)` |
| Renombrar columnas | `df.rename(columns={"viejo": "nuevo"})` |
| Limpiar texto | `df["col"].str.strip().str.lower()` |
| Unificar categorías | `df["col"].replace({"viejo": "nuevo"})` |
| Texto a número | `pd.to_numeric(df["col"].str.extract(r"(\d+)", expand=False))` |
| Faltantes | `df.isna().sum()`, `df.dropna()`, `df.fillna(valor)` |
| Duplicados | `df.duplicated().sum()`, `df.drop_duplicates()` |
| Apilar tablas | `pd.concat([df1, df2], ignore_index=True)` |
| Combinar por llave | `df1.merge(df2, left_on="a", right_on="b", how="left")` |
| Separar columna | `df["col"].str.split("-", expand=True)` |
| Filtrar filas | `df[df["col"] == x]`, `df.loc[condicion, columnas]`, `df.query("...")` |
| Rangos | `pd.cut(df["edad"], bins=[...], labels=[...])` |
| Agrupar | `df.groupby("col")["otra"].mean()`, `.agg(nombre=("col", "func"))` |
| Tabla cruzada | `pd.crosstab(df["a"], df["b"], normalize="index")` |
| Tabla dinámica | `df.pivot_table(index=..., columns=..., values=..., aggfunc="mean")` |
| Graficar | `df.plot(kind="bar")`, `ax.set_title(...)`, `ax.figure.savefig(ruta)` |
| Guardar | `df.to_csv(ruta, index=False)` |